# Garbage Classification based on images
### ENSF 617 - A2

Transfer learning with pretrained model [ResNet-50](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.resnet50.html#torchvision.models.resnet50)

Imports & Device Setup

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt
import os

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Configurations

In [ ]:
BATCH_SIZE = 32
FINE_TUNE_BATCH_SIZE = 16
NUM_WORKERS = 2
NUM_CLASSES = 4
EPOCHS = 10
UNFREEZE_EPOCH = 5
INITIAL_LR = 1e-3
FINE_TUNE_LR = 1e-5
SAVE_MODEL_PATH = "best_image_model.pth"

TRAIN_PATH = "path_to_train"
VAL_PATH = "path_to_val"
TEST_PATH = "path_to_test"


Data Transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

Load Dataset

In [ ]:
train_ds = ImageFolder(TRAIN_PATH, transform=train_transform)
val_ds   = ImageFolder(VAL_PATH, transform=test_transform)
test_ds  = ImageFolder(TEST_PATH, transform=test_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

ResNet Image Encoder Definition

In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self, pretrained=True, freeze_backbone=True, proj_dim=256):
        super().__init__()

        self.backbone = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT if pretrained else None
        )

        self.feature_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.projection = nn.Sequential(
            nn.Linear(self.feature_dim, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU()
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.projection(features)

Garbage Image Classifier Definition

In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self, encoder, num_classes=NUM_CLASSES):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        embeddings = self.encoder(x)
        return self.classifier(embeddings)

Initialize Model

In [ ]:
encoder = ImageEncoder(pretrained=True, freeze_backbone=True)
model = ImageClassifier(encoder).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=INITIAL_LR,
    weight_decay=1e-4
)

Training Loop Definition

In [ ]:
def train_model():
    best_val_loss = float("inf")

    for epoch in range(EPOCHS):

        # Unfreeze backbone
        if epoch == UNFREEZE_EPOCH:
            print("Unfreezing backbone...")
            for param in model.encoder.backbone.parameters():
                param.requires_grad = True

            optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=FINE_TUNE_LR,
                weight_decay=1e-4
            )

        # ---- Train ----
        model.train()
        running_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # ---- Validation ----
        model.eval()
        val_loss_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss_total += loss.item()

        val_loss = val_loss_total / len(val_loader)

        print(f"Epoch {epoch+1}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), SAVE_MODEL_PATH)
            print("Saved best model.")

Test Evaluation Definition

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def evaluate():
    model.load_state_dict(torch.load(SAVE_MODEL_PATH))
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    accuracy = (all_preds == all_labels).mean()
    print(f"\nTest Accuracy: {accuracy:.4f}\n")

    # ---- Classification Report ----
    class_names = test_ds.classes
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # ---- Confusion Matrix ----
    cm = confusion_matrix(all_labels, all_preds)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=class_names,
                yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.show()

Run

In [ ]:
train_model()
evaluate()